# Naive bayes

## Importación de librerías, paquetes y definición de constantes

In [1]:
#import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# Split para modelado
from sklearn.model_selection import train_test_split
# Extract
from sklearn.feature_extraction.text import CountVectorizer
# To save models
import json
import pickle
# Modelado
from sklearn.naive_bayes import MultinomialNB, BernoulliNB, GaussianNB
# Métricas
from utils import get_classifier_metrics
from sklearn.metrics import accuracy_score
# Optimizar
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.model_selection import RandomizedSearchCV


## Recopilación de datos

In [2]:
df = pd.read_csv("../data/raw/playstore_reviews.csv")
df.head()

,package_name,review,polarity
0,com.facebook.katana,privacy at least put some option appear offli...,0
1,com.facebook.katana,"messenger issues ever since the last update, ...",0
2,com.facebook.katana,profile any time my wife or anybody has more ...,0
3,com.facebook.katana,the new features suck for those of us who don...,0
4,com.facebook.katana,forced reload on uploading pic on replying co...,0


## Análisis descriptivo 

In [3]:
df

,package_name,review,polarity
0,com.facebook.katana,privacy at least put some option appear offli...,0
1,com.facebook.katana,"messenger issues ever since the last update, ...",0
2,com.facebook.katana,profile any time my wife or anybody has more ...,0
3,com.facebook.katana,the new features suck for those of us who don...,0
4,com.facebook.katana,forced reload on uploading pic on replying co...,0
...,...,...,...
886,com.rovio.angrybirds,loved it i loooooooooooooovvved it because it...,1
887,com.rovio.angrybirds,all time legendary game the birthday party le...,1
888,com.rovio.angrybirds,ads are way to heavy listen to the bad review...,0
889,com.rovio.angrybirds,fun works perfectly well. ads aren't as annoy...,1


In [4]:
df.shape

(891, 3)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   package_name  891 non-null    object
 1   review        891 non-null    object
 2   polarity      891 non-null    int64 
dtypes: int64(1), object(2)
memory usage: 21.0+ KB


In [6]:
df.nunique()

package_name     23
review          891
polarity          2
dtype: int64

## Limpieza de datos

### Eliminar duplicados:

In [7]:
df.duplicated().sum()

np.int64(0)

### No hay duplicados pero nos aseguramos de borrarlos

In [8]:
if df.duplicated().sum():
    df = df.drop_duplicates(keep='first')
print(df.shape)
df.head()

(891, 3)


,package_name,review,polarity
0,com.facebook.katana,privacy at least put some option appear offli...,0
1,com.facebook.katana,"messenger issues ever since the last update, ...",0
2,com.facebook.katana,profile any time my wife or anybody has more ...,0
3,com.facebook.katana,the new features suck for those of us who don...,0
4,com.facebook.katana,forced reload on uploading pic on replying co...,0


In [9]:
df.duplicated().sum()

np.int64(0)

### Valores nulos o faltantes:

In [10]:
df.isnull().sum().sort_values(ascending=False) / len(df)

package_name    0.0
review          0.0
polarity        0.0
dtype: float64

#### No hay valores nulos

### Limpieza de datos: Eliminar información irrelevante

#### Borramos la variable 'package_name' porque no influye en la polaridad de los comentarios, solo necesitamos el texto

In [11]:
df.drop(['package_name'], axis = 1, inplace = True)
df

,review,polarity
0,privacy at least put some option appear offli...,0
1,"messenger issues ever since the last update, ...",0
2,profile any time my wife or anybody has more ...,0
3,the new features suck for those of us who don...,0
4,forced reload on uploading pic on replying co...,0
...,...,...
886,loved it i loooooooooooooovvved it because it...,1
887,all time legendary game the birthday party le...,1
888,ads are way to heavy listen to the bad review...,0
889,fun works perfectly well. ads aren't as annoy...,1


## Split de datos


#### Limpiamos la cadena de string para que no tenga espacios y tenga solo minúsculas

In [12]:
df["review"] = df["review"].str.strip().str.lower()

In [13]:
X = df['review']
y = df['polarity']
X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size=0.2,
                                                    random_state=24)
X_test

169    keep calm and play temple run it's really fun....
556    doesn't login crap doesn't work. only app tat ...
206    needed to learn the hard way about town hall p...
71     really great! twitter is an amazing app which ...
677    ŕ¤°ŕ¤žŕ¤žŕľŕ¤°ŕľ ŕ¤ yo hamro keyboard chai ...
                             ...                        
6      major flaws constant updates and always gettin...
102    even though i am loving the new update, but th...
772    epaper not upto date epaper not updated for da...
738    some notification problem although i've enable...
654    very smooth i'd say it's much better than just...
Name: review, Length: 179, dtype: object

#### Vectorizamos la cadena de string

In [14]:
vec_model = CountVectorizer(stop_words = "english")
vec_model


,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None
,stop_words,'english'
,token_pattern,'(?u)\\b\\w\\w+\\b'
,ngram_range,"(1, ...)"
,analyzer,'word'


#### CountVectorizer necesita que se le pase una cadena de string para poder trabjar

In [15]:
X_train = vec_model.fit_transform(X_train).toarray()
X_test = vec_model.transform(X_test).toarray()
X_train

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(712, 3203))

## Generación del modelo Multinomial

In [16]:
naive_bayes_multi = MultinomialNB()
naive_bayes_multi.fit(X_train,y_train)
y_pred_test = naive_bayes_multi.predict(X_test)

In [17]:
y_pred_train = naive_bayes_multi.predict(X_train)

In [18]:
naive_bayes_multi.score(X_test, y_test)

0.8491620111731844

## Generación del modelo Bernoulli

In [19]:
naive_bayes_bernoulli = BernoulliNB()
naive_bayes_bernoulli.fit(X_train,y_train)
y_pred_test = naive_bayes_bernoulli.predict(X_test)

In [20]:
y_pred_train = naive_bayes_multi.predict(X_train)

In [21]:
naive_bayes_bernoulli.score(X_test, y_test)

0.7430167597765364

## Generación del modelo Gaussian

In [22]:
naive_bayes_gauss = GaussianNB()
naive_bayes_gauss.fit(X_train,y_train)
y_pred_test = naive_bayes_gauss.predict(X_test)

In [23]:
y_pred_train = naive_bayes_gauss.predict(X_train)

In [24]:
naive_bayes_gauss.score(X_test, y_test)

0.7597765363128491

## Optimización del modelo, escogemos el Multinomial porque es el que mejores métricas obtiene

### Parámetros para la cuadrícula

In [25]:
param_grid = {'alpha': [0.01, 0.1, 1, 1.5, 2, 5],
              'force_alpha': [True, False],
              'fit_prior': [True, False]              
    }
grid_search = GridSearchCV(estimator=naive_bayes_multi,
                           param_grid=param_grid,
                           cv=5,
                           refit='accuracy'
                           
)


In [26]:
grid_search.fit(X_train, y_train)

,estimator,MultinomialNB()
,param_grid,"{'alpha': [0.01, 0.1, ...], 'fit_prior': [True, False], 'force_alpha': [True, False]}"
,scoring,None
,n_jobs,None
,refit,'accuracy'
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,alpha,1.5


In [27]:
grid_search.best_params_

{'alpha': 1.5, 'fit_prior': False, 'force_alpha': True}

In [32]:
grid_multi = grid_search.best_estimator_
y_pred_test = grid_multi.predict(X_test)
y_pred_train = grid_multi.predict(X_train)

In [29]:
grid_multi.score(X_test,y_test)

0.8603351955307262

## Comparación de los modelos

In [30]:
metricas = {'Gaussian': naive_bayes_gauss.score(X_test, y_test),
            'Bernoulli': naive_bayes_bernoulli.score(X_test, y_test),
            'Multinomial': naive_bayes_multi.score(X_test, y_test),
            'Multinomial Optimizado': grid_multi.score(X_test,y_test)
}
metricas

{'Gaussian': 0.7597765363128491,
 'Bernoulli': 0.7430167597765364,
 'Multinomial': 0.8491620111731844,
 'Multinomial Optimizado': 0.8603351955307262}

### Observamos que el modelo Multinomial es el más preciso:

| Modelo                   | Precisión (%) |
|--------------------------|---------------|
| Gaussian                 | 75.98         |
| Bernoulli                | 74.30         |
| Multinomial              | 84.92         |
| Multinomial Optimizado  | 86.03         |

## Conclusiones

>- El mejor resultado lo obtenemos con el modelo multinomial y optimizando podemos conseguir una mejora a considerar, así que elegimos este modelo.


## Guardado de modelos

In [33]:
models = {'naive_bayes_multi' : naive_bayes_multi,
          'naive_bayes_bernoulli' : naive_bayes_bernoulli,
          'naive_bayes_gauss' : naive_bayes_gauss  
          }
with open('C:/Users/Simón/sayons-intro-ml-new/models/modelos-naive-bayes.pkl', 'wb') as file:
    pickle.dump(models, file)